# watsonx Lab — Guided Track

**AI Insights from a Trusted Data Catalog**

You'll create a trusted catalog and ask it questions **in plain English** — no
coding required. This notebook has every step pre-written. Work through the cells
**top to bottom**: click a cell, press **▶ Run**, read what it shows, move on.

**How a notebook works:** it's a page made of boxes called *cells*. Each cell has
a step already written in it. You press Run and it does that step. The only thing
you'll type is a question in plain English, in one cell near the end.

> **Full written instructions:** see `labs/non-technical/README.md`.
> Run the cells **in order** — each builds on the one before. If something looks
> wrong, start again from the first cell.

## Setup — just press Run

Run the cell below once. You don't need to read it — it gets the tools ready.

In [ ]:
# ── Setup — just press ▶ Run. You don't need to read this cell. ──
# It installs the tools and defines the few helpers the cells below use.
!pip install -q ibm-watsonx-ai prestodb pandas matplotlib

import pandas as pd, matplotlib.pyplot as plt
from ibm_watsonx_ai import APIClient, Credentials
from ibm_watsonx_ai.foundation_models import ModelInference

_state = {}   # holds your client + database connection after you sign in

def sign_in(api_key, project_id, presto_host, presto_user, presto_pwd,
            url='https://us-south.ml.cloud.ibm.com', presto_port=443):
    import prestodb
    _state['project_id'] = project_id
    _state['client'] = APIClient(Credentials(url=url, api_key=api_key),
                                 project_id=project_id)
    _state['presto'] = prestodb.dbapi.connect(
        host=presto_host, port=presto_port, user=presto_user,
        catalog='iceberg_data', schema='marine', http_scheme='https',
        auth=prestodb.auth.BasicAuthentication(presto_user, presto_pwd))
    print('✅ Signed in. SDK version:', _state['client'].version)

def run_sql(sql):
    cur = _state['presto'].cursor(); cur.execute(sql)
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(cur.fetchall(), columns=cols)

def ask(question):
    """Turn an English question into SQL (shown to you), run it, return a table."""
    granite = ModelInference(model_id='ibm/granite-3-8b-instruct',
        api_client=_state['client'], project_id=_state['project_id'],
        params={'decoding_method':'greedy', 'max_new_tokens':200})
    schema = ('cod_landings_nl(year INT, landings_tonnes DOUBLE, '
              'licensed_fishers INT, fishery_status VARCHAR)')
    prompt = (f"Translate the question into ONE Presto SQL query.\n"
              f"Table: iceberg_data.marine.{schema}\n"
              f"Rules: SELECT only, no DDL/DML, add LIMIT 1000. Return ONLY the SQL.\n"
              f"Question: {question}")
    sql = granite.generate_text(prompt=prompt).strip().strip('`').strip()
    # safety: only ever run a read-only SELECT
    low = sql.lower()
    assert low.startswith('select'), 'The AI did not return a SELECT; try rephrasing.'
    assert not any(k in low for k in [' insert',' update',' delete',' drop',
           ' alter',' create',' merge']), 'Blocked a non-read-only statement.'
    print('The AI wrote this query:\n', sql, '\n')
    return run_sql(sql)

def make_chart(df):
    """Draw a line chart from the AI's answer, marking the 1992 moratorium."""
    year_col = next((c for c in df.columns if c.lower() == 'year'), df.columns[0])
    value_cols = [c for c in df.columns
                  if c != year_col and pd.api.types.is_numeric_dtype(df[c])]
    y = 'landings_tonnes' if 'landings_tonnes' in value_cols else (value_cols or [df.columns[-1]])[0]
    d = df.sort_values(year_col)
    plt.figure(figsize=(10, 5))
    plt.plot(d[year_col], d[y], marker='o', ms=3)
    if d[year_col].min() <= 1992 <= d[year_col].max():
        plt.axvline(1992, color='crimson', ls='--')
        plt.text(1992, plt.ylim()[1]*0.9, ' 1992 moratorium', color='crimson')
    plt.title(f'{y} by {year_col}'); plt.xlabel(year_col); plt.ylabel(y)
    plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print('Setup done. Now run the "Sign in" cell below.')

## Sign in

Paste the values your instructor gives you between the quotes, then run the cell.
If it prints a green check with a version, you're ready. (The API key and project
ID come from watsonx.ai; the Presto values come from your query engine's *Connect*
panel in watsonx.data.)

In [ ]:
sign_in(
    api_key     = '...',   # your IBM Cloud API key
    project_id  = '...',   # your watsonx.ai project ID
    presto_host = '...',   # Presto engine host
    presto_user = '...',   # Presto user
    presto_pwd  = '...',   # Presto password
    url         = 'https://us-south.ml.cloud.ibm.com',  # your region
)

## Read your table

This reads the first rows of your catalogued table **straight from the catalog** —
nothing is uploaded and no copy is made. The AI studio is reading your governed
data where it lives.

In [ ]:
run_sql('SELECT year, landings_tonnes, licensed_fishers, fishery_status '
        'FROM cod_landings_nl ORDER BY year LIMIT 10')

## Search the PDF by meaning — done in the watsonx.ai UI

This step is point-and-click, not code. In your watsonx.ai project choose
**New asset → Vector index**, upload `docs/cod_background.pdf`, pick the offered
embedding model, and choose your watsonx.data **Milvus** as the store. Create the
index, then search it with a phrase like *"what happened to the fishery in the
early 1990s?"* — it finds the right passage by meaning, not exact words.

See `labs/non-technical/README.md` for the full click-by-click walkthrough. The
document stays inside your governed environment the whole time. Then come back
here for the payoff.

## Ask your question in plain English

Type a question between the quotes and run the cell. The AI turns your sentence
into a query, **shows you the exact query it wrote** (nothing hidden), runs it on
your data in watsonx.data, and brings back the answer as a table.

In [ ]:
question = 'Show cod landings in tonnes by year, ordered by year'
answer = ask(question)
answer

## Make the chart

Run this to turn the answer above into a line chart — the enormous late-1960s
peak, the steep fall, and the 1992 collapse.

In [ ]:
make_chart(answer)

## Try your own question

Change the sentence below to anything you like — for example
*"What's the single biggest year-over-year drop?"* — run it, then run
`make_chart(answer)` again to visualise the new result.

In [ ]:
question = "What's the single biggest year-over-year drop in landings?"
answer = ask(question)
answer

## Wrap-up

Step back and notice what just happened: you built the tools, catalogued the
data, and got an analysis and a chart **just by asking a question in plain
English** — and the AI even showed you the exact query it used. The data never
left your environment.

**Now imagine** the table wasn't public cod numbers but student records, patient
data, or unpublished research — and the PDF a confidential report. Every step
would be identical, and **nothing would have left your organization**. That's the
difference between *"I used an AI"* and *"I used AI responsibly on data I'm
accountable for."*

> ### ⚠️ Stop your engines
> You created your own query engine and vector engine, and they **bill while
> running**. In watsonx.data → Infrastructure manager, **stop both now** so they
> don't keep costing money after the session.